# In Silico Protein Digestion using Rapid Peptides Generator

Rapid Peptides Generator (RPG), is a standalone software dedicated to predict proteases-induced cleavage sites on sequences.

RPG is a python tool taking a (multi-)fasta/fastq file of proteins as input and digest each of them. The digestion mode can be either ‘concurrent’, i.e. all enzymes are present at the same time during digestion, or ‘sequential’. In sequential mode, each protein will be digested by each enzyme, one by one.

The resulting peptides contain information about positions of cleavage site, peptide sequences, length, mass as well as an estimation of isoelectric point (pI) of each peptide. Results are outputted in multi-fasta, CSV or TSV file.

# Setup
## Import and install Packages

In [63]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [64]:
import pandas as pd
import re
from functions import read_fastafile

In [65]:
# Display session information
session_info.show()

In [66]:
!pip install rpg

In [67]:
!rpg --help

usage: rpg [-h] [-a] [-b str] [-d str] [-e str [str ...]] [-f str] [-i path]
           [-l] [-s str] [-m float [float ...] | -t int [int ...]] [-n]
           [-x str [str ...]] [-z str [str ...]] [-y str] [-p str] [-w str]
           [-o str | -r] [-c int] [-q | -v] [--version]

This software takes protein sequences as input (-i option). All sequences will
be cleaved according to selected enzymes (-e option) and given miscleavage
percentage (-m option). Cleaving can be sequential or concurrent (-d option).
Resulting peptides are outputted in a file (-o option) in fasta, csv or tsv
format (-f option). Classical enzymes are included (-l option to print
available enzymes) but it is possible to define other enzymes (-a option). See
https://gitlab.pasteur.fr/nmaillet/rpg/ and https://rapid-peptide-
generator.readthedocs.io for more informations.

optional arguments:
  -h, --help            show this help message and exit
  -a, --addenzyme       Create a new enzyme. See https://rapid-pepti

## Set working directory

In [68]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/02_In-Silico_Digestion"):
    print("WARNING: The working directory is not set to the '02_In-Silico_Digestion'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_In-Silico_Digestion' folder (\"{cwd}\").")

Current working directory: /Users/lauranieba/Documents/Uni/Master_ETH/FS26/Lab_Rotation_Davos/Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_In-Silico_Digestion


In [69]:
# Data directory for fasta data
input_dir = "data/digestion_input"
output_dir = "data/digestion_output"
fasta_dir = "../01_UniProt/data/raw_data/fasta"

## Read in Fasta files

In [70]:
fasta_canonical = os.path.join(fasta_dir, 'uniprotkb_canonical_2026_03_31.fasta')
fasta_full_proteome = os.path.join(fasta_dir, 'uniprotkb_full_proteome_2026_03_31.fasta')
fasta_alternative_product = os.path.join(fasta_dir, 'uniprotkb_alternative_products_isoforms_2026_03_31.fasta')

# Performing digestion with RPG

In [71]:
#Show list of possible Enzymes to check which number is Trypsin (42)
!rpg -l

1: Arg-C
2: Asp-N
3: BNPS-Skatole
4: Bromelain
5: Caspase-1
6: Caspase-2
7: Caspase-3
8: Caspase-4
9: Caspase-5
10: Caspase-6
11: Caspase-7
12: Caspase-8
13: Caspase-9
14: Caspase-10
15: Chymotrypsin-high
16: Chymotrypsin-low
17: Clostripain
18: CNBr
19: Enterokinase
20: Factor-Xa
21: Ficin
22: Formic-acid
23: Glu-C
24: Glutamyl-endopeptidase
25: Granzyme-B
26: Hydroxylamine
27: Iodosobenzoic-acid
28: Lys-C
29: Lys-N
30: Neutrophil-elastase
31: NTCB
32: Papain
33: Pepsin-pH1.3
34: Pepsin-pH>=2
35: Proline-endopeptidase
36: Proteinase-K
37: Staphylococcal-peptidase-I
38: Thermolysin
39: Thrombin
40: Thrombin-SG
41: Tobacco-Etch-Virus
42: Trypsin
43: Asp-N Endopeptidase
44: ProAlanase
45: Elastase
46: aLP


Trial run with only canonical forms that have isoforms (~10'000 proteins).

For the digestion of the entire proteome I used Euler. The script that I used is shown below. The fragments are filtered so that we only keep the ones that are 7 amino acids long or longer.

In [72]:
digestion_full_proteome_unique = read_fastafile(os.path.join(output_dir, 'digestion_full_proteome_unique.fasta'))

In [73]:
digestion_full_proteome_filtered = digestion_full_proteome_unique[digestion_full_proteome_unique["len"] >= 7]
digestion_full_proteome_filtered = digestion_full_proteome_filtered[digestion_full_proteome_filtered["len"] <= 52]

In [74]:
digestion_full_proteome_filtered

,ID,seqID,seq,len
0,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,MGLEALVPLAMIVAIFLLLVDLMHR,25
3,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,YPPGPLPLPGLGNLLHVDFQNTPYCFDQLR,30
5,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,FGDVFSLQLAWTPVVVLNGLAAVR,24
7,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,GEDTADRPPAPIYQVLGFGPR,21
8,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,SQGVILSR,8
...,...,...,...,...
2647293,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,QNIFHFLYR,9
2647298,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,KPWFQLGSMNPR,12
2647322,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,SQIVLFQK,8
2647325,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,FHFFCSMSCR,10


In [75]:
#save filtered df as csv
digestion_full_proteome_filtered.to_csv(os.path.join(output_dir, 'digestion_full_proteome_filtered.csv'), index=False)

The digestion was redone with 20% miscleavage to get longer peptides that could potentially be used as unique peptides for isoforms that are truncated versions of the canonical protein.

In [76]:
digestion_m20_full_proteome_unique = read_fastafile(os.path.join(output_dir, 'digestion_m20_full_proteome_unique.fasta'))

In [77]:
digestion_m20_full_proteome_filtered = digestion_m20_full_proteome_unique[digestion_m20_full_proteome_unique["len"] >= 7]
digestion_m20_full_proteome_filtered = digestion_m20_full_proteome_filtered[digestion_m20_full_proteome_filtered["len"] <= 52]

In [78]:
digestion_m20_full_proteome_filtered

,ID,seqID,seq,len
0,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,MGLEALVPLAMIVAIFLLLVDLMHR,25
2,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,WAARYPPGPLPLPGLGNLLHVDFQNTPYCFDQLR,34
4,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,FGDVFSLQLAWTPVVVLNGLAAVR,24
6,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,GEDTADRPPAPIYQVLGFGPRSQGVILSR,29
7,A0A087X1C5,sp|A0A087X1C5|CP2D7_HUMAN,YGPAWREQR,9
...,...,...,...,...
2125926,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,IPLLRKPWFQLGSMNPR,17
2125929,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,SRIPLLR,7
2125945,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,NRSQIVLFQK,10
2125947,A6NHP3-2_Q495Y8-2,sp|A6NHP3-2_Q495Y8-2|SPE2B_SPDE2_HUMAN,RFHFFCSMSCR,11


In [79]:
#save filtered df as csv
digestion_m20_full_proteome_filtered.to_csv(os.path.join(output_dir, 'digestion_m20_full_proteome_filtered.csv'), index=False)